In [17]:
from pathlib import Path

from sl_shared_assets import SessionData, ProjectManifest, generate_project_manifest

from sl_forgery.utils.dataclass import ProjectData, TargetGroup
from sl_forgery.analysis.plotting import Plotting
from sl_forgery.analysis.processing import Processing

filter_path = Path(r"C:\Users\jacob\OneDrive\Desktop\PlaceFields\sl-forgery\src\sl_forgery\analysis\tm6_filter.yaml")
manifest_path = Path(r"C:\Users\jacob\OneDrive\Desktop\PlaceFields\slf_data\TM_06_pilot_manifest.feather")
work_dir = Path(r"C:\Users\jacob\OneDrive\Desktop\PlaceFields\slf_data")

data = ProjectData.create(project_name="TM_06_pilot", manifest_path=manifest_path, filter_path=filter_path, working_directory=work_dir)
cache = dict()

#Sessions
# Plotting.plot_session("single_day", 4, data.get_session("2025-06-23-13-32-06-980761"))

# Compute Umaps
# for session_name in data.manifest.get_sessions(animal=6):
#     Processing.compute_single_session_umap("single_day", data.get_session(session_name))

# Plot Umaps
# fig =Plotting.plot_umap("single_day", data.get_session("2025-06-23-13-32-06-980761"))
# Plotting.plot_all_single_session_umaps(target_group=TargetGroup.SINGLE_DAY, animal=data.get_mouse(6))

# Example of how to save a figure
# fig.write_html(r"C:\Users\jacob\OneDrive\Desktop\PlaceFields\sl-forgery\src\sl_forgery\analysis\umap.html")


In [ ]:
from enum import Enum

import numpy as np
from plotly import graph_objects as go

from sl_forgery.utils.dataclass import AnimalData, ProjectData, TargetGroup
from sl_forgery.analysis.processing import Processing, bin_size, cue_length, track_length


class TraceType(str, Enum):
    PER_TRIAL = "per_trial"
    AVERAGE = "average"
 

def plot_multi_session(
        target_group: str | TargetGroup,
        trace_type: str | TraceType,
        cell: int, 
        animal: AnimalData, 
        save_path: Path | None = None,
        ):
    
    if isinstance(target_group, str):
        target_group = TargetGroup(target_group)   

    if isinstance(trace_type, str):
        trace_type = TraceType(trace_type)


    frames = []


    # TODO: Task specific
    xaxis = np.arange(bin_size / 2, track_length, bin_size)
    cue_positions = range(0, track_length, cue_length * 2) # *2 bc of the gray region

    for session in animal.sessions:
        session_avg_df, sess_sem, result, trial_avg_df = Processing.bin_data(target_group, session, cache)
        
        match trace_type:
            case TraceType.PER_TRIAL:
                traces = [go.Scatter(
                    x = xaxis,
                    y = trial_avg_df[i, cell],
                    mode='lines',
                    line=dict(width=2),
                    name=f"Trial {i+1}",
                    visible=True,
                ) for i in range(result.shape[0])]

            case TraceType.AVERAGE:
                cell_val = session_avg_df[f'cell_{cell}_signal_binned'][-1]
                sem = sess_sem[cell]
                mean = cell_val.to_numpy()
                traces = []
                traces.append(go.Scatter(
                    x = xaxis,
                    y = mean,
                    mode='lines',
                    line=dict(width=3),
                    name="Mean"
                ))

                upper = mean + sem
                lower = mean - sem   

                # Add trace for the standard error of the mean
                traces.append(go.Scatter(
                    x=list(xaxis) + list(xaxis[::-1]),  # x followed by reversed x
                    y=list(upper) + list(lower[::-1]),  # upper followed by reversed lower
                    fill='toself',
                    fillcolor='rgba(0, 0, 255, 0.2)',  # RGBA for transparency
                    line=dict(color='rgba(255,255,255,0)'),  # No border
                    hoverinfo='skip',
                    name='SEM'
                )) 

        frames.append(go.Frame(
            data=traces,
            name=session.name
        ))

    fig = go.Figure(data=frames[0].data, frames=frames)

    slider_steps = [
        {
            'method': 'animate',
            'args': [[session.name], dict(mode='immediate', transition=dict(duration=0))],
            'label': ProjectData.parse_session(session.name),
        }
        for session in animal.sessions
    ]


    # Slider
    fig.update_layout(
        sliders=[
            {
                'active': 0,
                'steps': slider_steps,
            }
        ],
        updatemenus=[{
        'type': 'buttons',
        'y': -.15,
        'buttons': [
            {
                'label': 'Play',
                'method': 'animate',
                'args': [None, {'frame': {'duration': 1000, 'redraw': True}, 'transition': {'duration':1000}, 'fromcurrent': True}]
            },

        ]
    }]
    )

    # Axes
    fig.update_layout(
        title=dict(
            text=f"Cell Fluorescence Trial Averages",
            x=.5,
        ),
        plot_bgcolor='white',
        xaxis=dict(
            title="Track position (cm)",
            range=[0, track_length]
        ),
        yaxis=dict(
            title="Flourescant Signal",
            range=[0, 6000]
        ),
    )



    fig.update_layout(
        annotations=
        [
            *[
                dict(
                    text=f"Cue {i+1}",
                    xref="x", yref="paper",
                    x=pos + cue_length / 2, y=1, 
                    xanchor="center", yanchor="top",
                    align="center", 
                    showarrow=False,
                ) for i, pos in enumerate(cue_positions)          
            ],
        ],
        shapes=[
                dict(type="rect", x0=pos, x1=pos+cue_length, y0=0, y1=1, xref="x", yref="paper",
                    fillcolor="lightsteelblue", opacity=0.4, layer="below", line_width=0) 
                for pos in cue_positions
            ],
    )

    Plotting._clear_axes(fig)

    fig.show(renderer="browser")
    return fig

plot_multi_session("multi_day", cell=80, animal=data.get_mouse(6), trace_type=TraceType.AVERAGE)


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'line': {'width': 3},
              'mode': 'lines',
              'name': 'Mean',
              'type': 'scatter',
              'x': {'bdata': ('AAAAAAAABEAAAAAAAAAeQAAAAAAAAC' ... 'AAAABwbEAAAAAAABBtQAAAAAAAsG1A'),
                    'dtype': 'f8'},
              'y': {'bdata': ('+c2qRBCsfkDgK86IgaZ+QB6B9P+U6n' ... 'fbsLEYfkCB3EfG46N9QGbbYH/98n5A'),
                    'dtype': 'f8'}},
             {'fill': 'toself',
              'fillcolor': 'rgba(0, 0, 255, 0.2)',
              'hoverinfo': 'skip',
              'line': {'color': 'rgba(255,255,255,0)'},
              'name': 'SEM',
              'type': 'scatter',
              'x': [2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5,
                    52.5, 57.5, 62.5, 67.5, 72.5, 77.5, 82.5, 87.5, 92.5, 97.5,
                    102.5, 107.5, 112.5, 117.5, 122.5, 127.5, 132.5, 137.5, 142.5,
                    147.5, 152.5, 157.5, 162.5, 167.5, 172.5, 177.5, 182.5, 187.5,
                    192.5, 197.5, 202.5, 207.5, 212.5, 217.5, 222.5, 227.5, 232.5,
                    237.5, 237.5, 232.5, 227.5, 222.5, 217.5, 212.5, 207.5, 202.5,
                    197.5, 192.5, 187.5, 182.5, 177.5, 172.5, 167.5, 162.5, 157.5,
                    152.5, 147.5, 142.5, 137.5, 132.5, 127.5, 122.5, 117.5, 112.5,
                    107.5, 102.5, 97.5, 92.5, 87.5, 82.5, 77.5, 72.5, 67.5, 62.5,
                    57.5, 52.5, 47.5, 42.5, 37.5, 32.5, 27.5, 22.5, 17.5, 12.5,
                    7.5, 2.5],
              'y': [498.4742525996449, 497.0248492822292, 521.9413712872114,
                    527.0282550208793, 483.94215104256256, 477.1762077089962,
                    464.2984572069522, 467.68466289589696, 465.260220464693,
                    474.2120191559449, 486.6025894944599, 494.9877896038469,
                    490.33719129991255, 484.22234977115716, 480.0635422009204,
                    489.7736341696508, 506.7050589048431, 510.2237056435617,
                    538.8266268158801, 555.0344191427715, 573.1333020047904,
                    573.0431701931179, 530.6193797820031, 503.0150755344052,
                    510.33931937926667, 507.17830996007024, 496.4551162634188,
                    488.6445324581945, 493.11764137953736, 499.6418311168714,
                    507.95420675527237, 539.2986733746085, 572.7763370673351,
                    585.7591540200274, 591.5344817012258, 612.3347013127005,
                    700.8608461867268, 820.1332192914768, 1012.5287266943851,
                    900.5276196187124, 667.7019668754451, 527.0921417728935,
                    489.16703584069876, 473.89126217860746, 483.1069419239787,
                    491.0714370805121, 481.4820479798065, 502.3633394682889,
                    488.0104386428785, 467.0041700543815, 472.0153260269265,
                    466.5514277093749, 460.4384975879493, 472.934139014777,
                    500.94717526484254, 590.5229152835211, 767.6592764443578,
                    867.7758177892217, 735.4251398182652, 637.264113138963,
                    569.986652398407, 559.1769089010895, 551.7140646539358,
                    539.9790725798555, 515.6304550280453, 491.8839914202,
                    486.14478763368305, 478.6767415021836, 474.9472996443242,
                    483.27383410316605, 490.95702299806817, 489.7082299666833,
                    484.21064205222643, 502.7540985943427, 530.4814488100758,
                    535.357791733671, 527.8013150041082, 516.588448102228,
                    492.90772475261, 481.1545032831432, 477.26992240684604,
                    466.9880919900581, 468.72658216167724, 471.2841212206283,
                    475.9175588001081, 468.9636520104936, 459.24775709496333,
                    451.55501324247973, 453.89566669459975, 449.47003050650744,
                    456.79772544004, 457.2111903497625, 465.8442573179437,
                    467.38138253338946, 483.7883999345314, 483.0336908726554]}],
    'frames': [{'data'